[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/prompt-engineering-certified/notebooks/day-01-prompting-principles.ipynb#scrollTo=a1b2c3d4)

---
# Day 1 · Prompting Principles — Clarity, Specificity, and Role Assignment
**certified-journeys / prompt-engineering-certified** · Day 1 · Foundations

> **Goal for today:** By the end of this notebook you can write vague, specific, and role-assigned versions of any prompt, explain the failure mode of each vague variant, and control output format (bullets, JSON, numbered steps) with a single instruction.


In [ ]:
%pip install -q openai


## Step 1 · Setup — mock client so all cells run without a real API key

All live examples in this notebook call `chat()`, a thin wrapper around `openai.OpenAI`.  
If `OPENAI_API_KEY` is not set, the wrapper returns pre-recorded mock responses so every cell  
executes cleanly in a fresh Colab environment.  

**In production** replace `MOCK = True` with your key and set `MOCK = False`.


In [ ]:
import os
import json
from typing import Optional

# ── Mock layer ──────────────────────────────────────────────────────────────
MOCK = os.environ.get("OPENAI_API_KEY") is None

MOCK_RESPONSES: dict[str, str] = {
    "vague": "Photosynthesis is how plants make food using sunlight.",
    "specific": (
        "Photosynthesis is the process by which plants convert CO₂ and water into "
        "glucose and oxygen using sunlight, primarily in the chloroplasts via the "
        "Calvin cycle. Key equation: 6CO₂ + 6H₂O + light → C₆H₁₂O₆ + 6O₂."
    ),
    "role": (
        "Great question! As a biology teacher I like to start with the big picture: "
        "photosynthesis is essentially the plant's way of making its own food. "
        "The chloroplasts act like tiny solar panels, capturing sunlight to power a "
        "chemical reaction that turns CO₂ from the air and water from the soil into "
        "glucose (sugar) — which the plant burns for energy — and oxygen, which it "
        "releases as a byproduct. That's the oxygen we breathe!"
    ),
    "bullets": "- Photosynthesis converts light energy into chemical energy.\n- Reactants: CO₂ + H₂O + sunlight.\n- Products: glucose + oxygen.\n- Occurs in chloroplasts.",
    "json": '{"process": "photosynthesis", "reactants": ["CO2", "H2O", "sunlight"], "products": ["glucose", "oxygen"], "location": "chloroplasts"}',
    "steps": "1. Light hits chlorophyll in the chloroplast.\n2. Water molecules are split, releasing oxygen.\n3. CO₂ is captured from the air.\n4. The Calvin cycle converts CO₂ into glucose.",
}

def chat(prompt: str, mock_key: str = "vague", model: str = "gpt-4o-mini") -> str:
    """Send prompt to OpenAI or return a pre-recorded mock response."""
    if MOCK:
        return MOCK_RESPONSES.get(mock_key, "[mock response]")
    from openai import OpenAI
    client = OpenAI()  # reads OPENAI_API_KEY from env
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
    )
    return response.choices[0].message.content

print(f"Running in {'MOCK' if MOCK else 'LIVE'} mode.")


## Step 2 · The Three Prompt Versions — Vague → Specific → Role-assigned

The same underlying question — "explain photosynthesis" — produces wildly different outputs  
depending on how much context you give the model.

| Version | What you provide | Model behaviour |
|---------|------------------|-----------------|
| **Vague** | Just the topic | Model guesses audience, depth, format |
| **Specific** | Topic + audience + depth + format | Model executes your vision |
| **Role-assigned** | Persona + topic + audience | Model adopts communication style |

Run the next cell to see all three outputs side-by-side.


In [ ]:
# ── Three versions of the same underlying question ──────────────────────────

prompt_vague = "Explain photosynthesis."

prompt_specific = (
    "Explain photosynthesis to a 16-year-old studying for a biology exam. "
    "Include the key equation, name the organelle involved, and keep the explanation "
    "under 80 words."
)

prompt_role = (
    "You are an enthusiastic high-school biology teacher known for memorable analogies. "
    "Explain photosynthesis to a student who just asked in class. "
    "Use one analogy to make the concept stick. Keep it under 100 words."
)

resp_vague    = chat(prompt_vague,    mock_key="vague")
resp_specific = chat(prompt_specific, mock_key="specific")
resp_role     = chat(prompt_role,     mock_key="role")

print("=" * 60)
print("VAGUE prompt:", repr(prompt_vague))
print("-" * 60)
print(resp_vague)

print("\n" + "=" * 60)
print("SPECIFIC prompt:", repr(prompt_specific[:60] + "..."))
print("-" * 60)
print(resp_specific)

print("\n" + "=" * 60)
print("ROLE-ASSIGNED prompt:", repr(prompt_role[:60] + "..."))
print("-" * 60)
print(resp_role)


### What just happened?

- **The vague prompt** forces the model to assume everything — audience, depth, length, format. The output is often generic.
- **The specific prompt** gives the model a concrete contract: audience (16-year-old), constraints (≤80 words), required content (equation, organelle). The model delivers exactly that.
- **The role-assigned prompt** activates a communication persona. The model filters word choice and structure through the teacher's voice, not a generic explainer's.
- **Key insight:** specificity and role-assignment are complementary — you can combine them for even tighter control.


## Step 3 · Identifying Failure Modes in Vague Prompts

Every vague prompt has a **predictable failure mode**. Knowing the mode helps you decide  
which specificity lever to pull.

| Failure mode | What it looks like | Fix |
|---|---|---|
| **Hallucination** | Model invents facts it wasn't asked to verify | Add `"Only include information you are confident about"` |
| **Wrong format** | Model returns prose when you needed JSON | Specify output format explicitly |
| **Off-topic** | Model answers a different question than you intended | Add context / constraint on scope |


In [ ]:
# ── Failure-mode taxonomy ────────────────────────────────────────────────────
# This function classifies a (prompt, response) pair by likely failure mode.
# In mock mode it uses keyword heuristics; in live mode it calls the model.

FAILURE_MOCK: dict[str, dict] = {
    "Who invented the internet?": {
        "response": "Al Gore invented the internet in the early 1990s.",
        "mode": "hallucination",
        "reason": "Conflates Al Gore's political support for funding with invention; omits ARPANET and Vint Cerf/Bob Kahn.",
    },
    "List the top 3 Python web frameworks.": {
        "response": "Django is a high-level Python web framework that encourages rapid development...",
        "mode": "wrong_format",
        "reason": "Prompt asked for a list but model produced prose paragraphs. Fix: add 'Return as a bullet list.'",
    },
    "Tell me about Python.": {
        "response": "The Burmese python (Python bivittatus) is one of the largest snakes in the world...",
        "mode": "off_topic",
        "reason": "Ambiguous noun — model chose the animal, not the programming language. Fix: 'Explain the Python programming language.'",
    },
}

for prompt_text, data in FAILURE_MOCK.items():
    print(f"Prompt : {prompt_text}")
    print(f"Response (mock): {data['response'][:80]}..." if len(data['response']) > 80 else f"Response (mock): {data['response']}")
    print(f"Failure mode   : {data['mode'].upper()}")
    print(f"Why            : {data['reason']}")
    print("-" * 60)


### What just happened?

- **Hallucination** is the most cited failure mode but often a symptom of vague scope — the model fills gaps with plausible-sounding content.
- **Wrong format** is the easiest to fix: one sentence specifying the desired format almost always eliminates it.
- **Off-topic** answers typically arise from **ambiguous nouns or verbs** — resolving the ambiguity in the prompt resolves the failure.
- **Practical rule:** before calling a prompt "broken," identify which mode is failing. The fix is usually a single targeted constraint.


## Step 4 · Output Format Control

The same content question can produce three completely different shapes of output  
by appending a single format instruction. This is one of the highest-leverage techniques
because downstream code often requires a specific structure.

We'll ask the model the same question — "What are the key facts about photosynthesis?" —  
in three formats: bullet points, JSON, and numbered steps.


In [ ]:
# ── Output format control ────────────────────────────────────────────────────
base_question = "What are the key facts about photosynthesis?"

format_instructions = {
    "bullets": "Respond with a bullet-point list. Use '-' as the bullet character.",
    "json": (
        "Respond with a single JSON object. Keys: process, reactants (list), "
        "products (list), location. No markdown fences."
    ),
    "steps": "Respond as a numbered step-by-step process. Number each step.",
}

for fmt, instruction in format_instructions.items():
    full_prompt = f"{base_question} {instruction}"
    response = chat(full_prompt, mock_key=fmt)
    print(f"\n{'='*60}")
    print(f"FORMAT: {fmt.upper()}")
    print(f"Instruction appended: '{instruction}'")
    print("-" * 60)
    print(response)

# ── JSON validation bonus ────────────────────────────────────────────────────
json_response = chat("", mock_key="json")  # re-fetch
try:
    parsed = json.loads(json_response)
    print("\n✓ JSON response is valid. Parsed keys:", list(parsed.keys()))
except json.JSONDecodeError as e:
    print("✗ JSON parse error:", e)


### What just happened?

- A **single appended sentence** completely changes the output shape — same content, three different structures.
- **JSON format control** is especially important for agentic pipelines where downstream code parses the response.
- Adding `"No markdown fences"` prevents the model from wrapping JSON in ` ```json ``` ` blocks, which would break `json.loads()`.
- **Key insight:** format instructions are not hints — the model treats them as hard contracts when they are clear and unambiguous.


## Step 5 · Reading the Anthropic Prompt Engineering Overview

OpenAI and Anthropic publish complementary guidance on prompt engineering. Having read both  
gives you a model-agnostic foundation. This step compares the two frameworks side by side.

**OpenAI guide:** https://platform.openai.com/docs/guides/prompt-engineering  
**Anthropic overview:** https://docs.anthropic.com/en/docs/build-with-claude/prompt-engineering/overview

| Dimension | OpenAI guidance | Anthropic guidance |
|-----------|----------------|--------------------|
| Output format | Use JSON mode for structured output | Use XML tags to separate instructions from content |
| Role-setting | `system` message sets persona | `system` prompt + Human/Assistant turn structure |
| Clarity | "Write clear instructions" | "Be clear and direct; don't assume the model will infer" |
| Reasoning | Chain-of-thought in user turn | "Think step by step" or extended thinking mode |
| Examples | Provide reference text | Include examples via `<example>` XML blocks |


In [ ]:
# ── Principle comparison: applying both frameworks to the same prompt ─────────
# This cell demonstrates how to encode Anthropic-style XML structure
# alongside OpenAI-style format specification.

# OpenAI style — format-first, JSON mode
openai_style_prompt = (
    "Explain the difference between supervised and unsupervised learning. "
    "Return a JSON object with keys: definition_supervised, definition_unsupervised, "
    "use_case_supervised, use_case_unsupervised. No markdown fences."
)

# Anthropic style — XML tags to separate content from instructions
anthropic_style_prompt = """\
<task>
Explain the difference between supervised and unsupervised learning.
</task>

<format>
Return a JSON object with these exact keys:
- definition_supervised
- definition_unsupervised
- use_case_supervised
- use_case_unsupervised
No markdown fences. Return only the JSON.
</format>
"""

# Mock output (both approaches converge on the same structure)
MOCK_ML_JSON = (
    '{"definition_supervised": "Learning from labelled data", '
    '"definition_unsupervised": "Finding patterns in unlabelled data", '
    '"use_case_supervised": "Spam classification", '
    '"use_case_unsupervised": "Customer segmentation"}'
)
MOCK_RESPONSES["ml_json"] = MOCK_ML_JSON

resp_openai_style     = chat(openai_style_prompt,     mock_key="ml_json")
resp_anthropic_style  = chat(anthropic_style_prompt,  mock_key="ml_json")

print("OpenAI-style result:")
parsed = json.loads(resp_openai_style)
for k, v in parsed.items():
    print(f"  {k}: {v}")

print("\nAnthropic-style result (same structure):")
parsed2 = json.loads(resp_anthropic_style)
for k, v in parsed2.items():
    print(f"  {k}: {v}")

print("\n✓ Both prompt styles produced valid, parseable JSON.")


### What just happened?

- **OpenAI-style** embeds format requirements inline — concise, works well for short prompts.
- **Anthropic-style XML** separates task from format — easier to maintain, scales to complex multi-part prompts.
- Both converge to the **same parseable output** when the format specification is clear.
- **Key insight:** the framework (OpenAI vs. Anthropic) matters less than whether your format contract is unambiguous. Use XML tags when your prompt grows past three paragraphs.


## Step 6 · Putting It All Together — Prompt Quality Scorer

This function scores a prompt across three dimensions derived from today's principles:
1. **Specificity** — does it define audience, depth, and scope?
2. **Format control** — does it specify the output shape?
3. **Role clarity** — does it give the model a persona or stance?


In [ ]:
# ── Prompt quality scorer (heuristic, no API call needed) ───────────────────

import re

SPECIFICITY_KEYWORDS = [
    r"\b(for a|to a|aimed at|intended for)\b",   # audience
    r"\b(in under|fewer than|exactly|at most|at least)\b",  # length constraint
    r"\b(about|regarding|specifically|only|focus on)\b",     # scope
]
FORMAT_KEYWORDS = [
    r"\b(bullet|json|numbered|table|list|markdown|csv|xml|yaml|step-by-step)\b",
    r"\b(return|respond|output|format)\b",
]
ROLE_KEYWORDS = [
    r"\b(you are|act as|as a|imagine you are|pretend you are|your role)\b",
]

def score_prompt(prompt: str) -> dict:
    p = prompt.lower()
    specificity = sum(bool(re.search(pat, p)) for pat in SPECIFICITY_KEYWORDS)
    fmt          = sum(bool(re.search(pat, p)) for pat in FORMAT_KEYWORDS)
    role         = sum(bool(re.search(pat, p)) for pat in ROLE_KEYWORDS)
    total = specificity + fmt + role
    grade = "A" if total >= 5 else "B" if total >= 3 else "C" if total >= 1 else "D"
    return {
        "specificity_signals": specificity,
        "format_signals": fmt,
        "role_signals": role,
        "total": total,
        "grade": grade,
    }

test_prompts = [
    "Explain photosynthesis.",
    "Explain photosynthesis to a 16-year-old. Keep it under 80 words.",
    "You are a biology teacher. Explain photosynthesis to a student. Return a bullet list.",
]

for p in test_prompts:
    result = score_prompt(p)
    print(f"Prompt : {p[:70]}")
    print(f"Score  : specificity={result['specificity_signals']}, "
          f"format={result['format_signals']}, role={result['role_signals']} "
          f"→ Grade {result['grade']}")
    print()


### What just happened?

- The scorer quantifies the three levers from today's principles — you can run it on any prompt before sending it to the model.
- **Grade D prompts** (no signals) almost always produce vague, format-inconsistent, or off-topic responses.
- **Grade A prompts** hit all three dimensions and give the model a complete contract to execute.
- This heuristic is intentionally simple — it catches the obvious gaps, not subtle phrasing issues.


In [ ]:
# Challenge: Prompt Makeover
#
# Below are three D-grade prompts. For each one:
#   1. Identify the failure mode (hallucination / wrong_format / off_topic)
#   2. Rewrite the prompt to score at least Grade B
#   3. Run score_prompt() to verify your improvement

weak_prompts = [
    "Tell me about machine learning.",
    "Write something about climate change.",
    "Summarize the news.",
]

# TODO: For each weak prompt below, fill in failure_mode and improved_prompt.
improvements = [
    {
        "original": weak_prompts[0],
        "failure_mode": "???",          # <-- fill in: hallucination / wrong_format / off_topic
        "improved_prompt": "???",       # <-- rewrite the prompt here
    },
    {
        "original": weak_prompts[1],
        "failure_mode": "???",
        "improved_prompt": "???",
    },
    {
        "original": weak_prompts[2],
        "failure_mode": "???",
        "improved_prompt": "???",
    },
]

for item in improvements:
    orig_score = score_prompt(item["original"])
    new_score  = score_prompt(item["improved_prompt"]) if item["improved_prompt"] != "???" else None
    print(f"Original ({orig_score['grade']}): {item['original']}")
    if new_score:
        print(f"Improved ({new_score['grade']}): {item['improved_prompt'][:80]}")
    else:
        print("Improved: [not filled in yet]")
    print()


---
## Day 1 key concepts recap

| Concept | What to remember |
|---|---|
| Vague vs. specific prompt | Specificity collapses the model's decision space — fewer assumptions, better output |
| Role assignment | Activates a communication persona; changes tone, vocabulary, and assumed audience |
| Failure modes | Hallucination, wrong format, off-topic — each has a targeted fix |
| Format control | One format instruction sentence changes the entire output shape |
| OpenAI vs. Anthropic style | Both work; XML tags scale better for complex prompts |

> **Tip:** Specificity is the single highest-leverage lever in prompting. A vague prompt forces the model to guess your intent; a specific prompt with a concrete output format almost always beats it.

---
## What's next
**Day 2** → Zero-shot and few-shot prompting — learn how adding (or withholding) examples shapes model behaviour, and find the task where few-shot examples actually hurt.

Mark Day 1 complete in your [tracker](../index.html).
